# Natural Language to Cypher with Ollama

This notebook demonstrates how to translate natural language questions into **Cypher** queries
using a locally-hosted LLM via **Ollama**. Cypher is the query language for **Neo4j** and other
labeled property graph (LPG) databases.

The pipeline uses:
- **LPG schema injection** (node labels, properties, relationship types)
- **Pydantic structured output** with Ollama's grammar-constrained decoding
- **Neo4j `EXPLAIN` validation** to catch syntax errors without executing
- A **self-correction retry loop** for robust query generation

## Prerequisites

1. Install and run [Ollama](https://ollama.com/) or use an external Ollama endpoint.
2. Pull a suitable model: `ollama pull qwen2.5:1.5b`
3. Install Python dependencies: `pip install ollama pydantic neo4j`
4. (Optional) Run a Neo4j instance for validation and execution

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install ollama pydantic neo4j

## Step 0: Connect to Ollama Endpoint
If you are using an Ollama endpoint, either Ollama Cloud or a dedicated server, you need to set the endpoint's base URL and your APU key.
1. Save your API key in a file outside of your project directory, that you can load into the notebook. **Never include API keys directly into your notebooks or program code!**
2. Define the variables `OLLAMA_BASE_URL` and `OLLAMA_API_KEY` to your configuration.

The following code tests your Ollama connection by listing your installed models.

In [ ]:
import os
OLLAMA_BASE_URL="https://gpu-01.insight.gsu.edu:11443"
OLLAMA_API_KEY=open(os.path.expanduser("~/.secrets/insight_ollama_msa8700_student.txt"), 'r').read().strip()
print(f"Length of API Key: {len(OLLAMA_API_KEY)}")

from ollama import Client

client = Client(
    host=OLLAMA_BASE_URL,
    headers={'Authorization': f'Bearer {OLLAMA_API_KEY}'}
)

import pandas as pd
import json
response = client.list()
print(f"Number of models: {len(response.models):,}")
# models = pd.DataFrame.from_records([m.model_dump() for m in response.models])
# for col in ['parent_model', 'format', 'family', 'families', 'parameter_size', 'quantization_level']:
#     models[col] = models.apply(lambda m: m.details[col], axis=1)

# display(models[['model', 'size', 'family', 'parameter_size', 'quantization_level']])

## Step 1: Define the LPG Schema

Unlike RDF/SPARQL which uses an OWL ontology, Cypher uses a **labeled property graph** schema.
We list every node label with its properties (and types), and every relationship type with its
direction and properties. The model can only reference what it sees — missing schema elements
lead to hallucinated labels and property names.

In [ ]:
LPG_SCHEMA = """
Node Labels:
- Person(id: STRING, name: STRING, birthYear: INT, role: STRING)
- Organization(id: STRING, name: STRING, industry: STRING, founded: INT)
- Publication(id: STRING, title: STRING, year: INT, doi: STRING)

Relationship Types:
- (Person)-[:WORKS_AT {since: INT, position: STRING}]->(Organization)
- (Person)-[:AUTHORED {contribution: STRING}]->(Publication)
- (Person)-[:COLLABORATES_WITH]->(Person)
- (Organization)-[:FUNDED]->(Publication)
"""

print("Schema loaded.")
print("Node labels: Person, Organization, Publication")
print("Relationships: WORKS_AT, AUTHORED, COLLABORATES_WITH, FUNDED")

## Step 2: Define the Structured Output Schema

The Pydantic model captures the generated Cypher query along with metadata:
- `query_intent` — MATCH, AGGREGATION, PATH, or EXISTENCE
- `node_labels_used` and `relationships_used` — for observability
- `explanation` — chain-of-thought description

In [ ]:
from pydantic import BaseModel


class CypherResult(BaseModel):
    query: str                    # Full executable Cypher query
    query_intent: str             # "MATCH" | "AGGREGATION" | "PATH" | "EXISTENCE"
    node_labels_used: list[str]   # e.g. ["Person", "Organization"]
    relationships_used: list[str] # e.g. ["WORKS_AT", "AUTHORED"]
    explanation: str              # One-sentence description

## Step 3: Build the System Prompt

The prompt includes the model's role, the graph schema, and two few-shot examples
demonstrating the `MATCH...WHERE...RETURN` pattern that Cypher uses.

In [ ]:
CYPHER_SYSTEM_PROMPT = f"""You are an expert Neo4j Cypher query generator.
Given a graph schema and a natural language question, generate a valid, executable Cypher query.

Rules:
- Use ONLY the node labels, properties, and relationship types defined in the schema
- Use MATCH...WHERE...RETURN structure
- Prefer OPTIONAL MATCH for potentially absent relationships
- Use WITH for intermediate aggregations
- Use ORDER BY and LIMIT where appropriate
- Do not include markdown formatting in the query field

Graph Schema:
{LPG_SCHEMA}

Examples:
Q: Find all people who work in the technology industry.
Cypher:
MATCH (p:Person)-[:WORKS_AT]->(o:Organization)
WHERE o.industry = "technology"
RETURN p.name, p.role, o.name AS organization

Q: Which people have authored publications?
Cypher:
MATCH (p:Person)-[:AUTHORED]->(pub:Publication)
RETURN DISTINCT p.name, collect(pub.title) AS publications
"""

## Step 4: Cypher Validation

### Option A: Validate with Neo4j `EXPLAIN`

Neo4j's `EXPLAIN` prefix parses and plans a query **without executing it** — zero data is read
or written. It returns detailed parse errors including line/column positions.

### Option B: Offline Validation (no Neo4j required)

For environments without a running Neo4j instance, we provide a basic keyword-based check
as a lightweight fallback.

In [ ]:
import re


def validate_cypher_offline(query: str) -> tuple[bool, str]:
    """Basic offline Cypher validation (no Neo4j required).

    Checks for common structural issues. For full validation,
    use Neo4j EXPLAIN (see CypherValidator class below).
    """
    query_stripped = query.strip()

    if not query_stripped:
        return False, "Empty query"

    # Check for required MATCH or RETURN keywords
    upper = query_stripped.upper()
    if "MATCH" not in upper and "RETURN" not in upper:
        return False, "Query must contain MATCH and/or RETURN clauses"

    # Check for balanced parentheses and brackets
    if query_stripped.count("(") != query_stripped.count(")"):
        return False, "Unbalanced parentheses"
    if query_stripped.count("[") != query_stripped.count("]"):
        return False, "Unbalanced brackets"
    if query_stripped.count("{") != query_stripped.count("}"):
        return False, "Unbalanced braces"

    return True, "Valid (offline check)"


# Test
print(validate_cypher_offline("MATCH (p:Person) RETURN p.name"))
print(validate_cypher_offline("MATCH (p:Person RETURN p.name"))

In [ ]:
# Full Neo4j EXPLAIN validator (requires a running Neo4j instance)

from neo4j import GraphDatabase
from neo4j.exceptions import CypherSyntaxError


class CypherValidator:
    """Validates Cypher queries using Neo4j's EXPLAIN command."""

    def __init__(
        self,
        uri: str = "bolt://localhost:7687",
        user: str = "neo4j",
        password: str = "password",
    ):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def validate(self, query: str) -> tuple[bool, str]:
        """Parse and plan the query without executing it."""
        try:
            with self.driver.session() as session:
                session.run(f"EXPLAIN {query}").consume()
            return True, "Valid"
        except CypherSyntaxError as e:
            return False, str(e)
        except Exception as e:
            # Fail open on connectivity issues
            print(f"  Warning: Neo4j unreachable, skipping validation: {e}")
            return True, "Skipped (no connection)"

    def run(self, query: str) -> list[dict]:
        """Execute a Cypher query and return results."""
        with self.driver.session() as session:
            result = session.run(query)
            return [dict(record) for record in result]

    def close(self):
        self.driver.close()

## Step 5: Generation with Self-Correction Loop

The generation function uses the offline validator by default. If you have a Neo4j instance
running, pass a `CypherValidator` instance for full `EXPLAIN`-based validation.

In [ ]:
MODEL = "qwen2.5:1.5b"


def nl_to_cypher(
    question: str,
    validator=None,
    max_retries: int = 3,
) -> CypherResult:
    """Translate a natural language question to a validated Cypher query.

    Args:
        question: Natural language question to translate.
        validator: Optional CypherValidator instance for Neo4j EXPLAIN validation.
                   Falls back to offline validation if None.
        max_retries: Maximum number of generation attempts.
    """
    validate_fn = (
        validator.validate if validator else validate_cypher_offline
    )
    error_context = ""

    for attempt in range(1, max_retries + 1):
        user_content = f"Question: {question}"
        if error_context:
            user_content += (
                f"\n\nYour previous attempt produced an invalid Cypher query.\n"
                f"Error: {error_context}\n"
                f"Please correct the query."
            )

        response = client.chat(
            model=MODEL,
            messages=[
                {"role": "system", "content": CYPHER_SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
            ],
            format=CypherResult.model_json_schema(),
            options={"temperature": 0.0},
        )

        result = CypherResult.model_validate_json(response.message.content)
        is_valid, error = validate_fn(result.query)

        status = "Valid" if is_valid else f"Error: {error}"
        print(f"  [Attempt {attempt}] {status}")

        if is_valid:
            return result
        error_context = error

    raise ValueError(f"Failed to generate valid Cypher after {max_retries} attempts.")

## Step 6: Run Examples

Let's test the pipeline with several questions against our person/organization/publication
graph schema.

In [ ]:
questions = [
    "Who are all the people working at organizations in the AI industry?",
    "Find all publications authored by people who collaborate with each other.",
    "Which organizations have funded more than 5 publications since 2020?",
    "List all people and their positions at their organizations.",
    "Who has collaborated with someone who works at a healthcare organization?",
]

In [ ]:
for q in questions:
    print(f"\n{'=' * 70}")
    print(f"Question: {q}")
    print("-" * 70)

    result = nl_to_cypher(q)

    print(f"\nIntent        : {result.query_intent}")
    print(f"Node Labels   : {result.node_labels_used}")
    print(f"Relationships : {result.relationships_used}")
    print(f"Explanation   : {result.explanation}")
    print(f"\nGenerated Cypher:\n{result.query}")

## Step 7: Inspect Structured Output Detail

Let's examine the full JSON structure for a single query.

In [ ]:
import json

result = nl_to_cypher(
    "Which organizations have funded publications authored by the same person?"
)
print(json.dumps(result.model_dump(), indent=2))

## Step 8 (Optional): Execute Against a Live Neo4j Instance

If you have Neo4j running, you can validate with `EXPLAIN` and execute queries.

Start Neo4j with Docker:
```bash
docker run -d --name neo4j \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/your-password \
  neo4j:latest
```

In [ ]:
# Uncomment and configure to use Neo4j EXPLAIN validation and execution

# validator = CypherValidator(
#     uri="bolt://localhost:7687",
#     user="neo4j",
#     password="your-password",
# )
#
# try:
#     result = nl_to_cypher(
#         "Find all people and their publications.",
#         validator=validator,
#     )
#     print(f"Query:\n{result.query}")
#
#     # Execute the query
#     rows = validator.run(result.query)
#     for row in rows:
#         print(row)
# finally:
#     validator.close()

## Key Takeaways

1. **LPG schema injection** lists node labels, properties, and relationship types — the Cypher equivalent of DDL for SQL or TBox for SPARQL.
2. **Neo4j's `EXPLAIN`** command validates syntax without executing — zero data is read or written, making it safe for validation loops.
3. **Offline validation** provides a lightweight fallback when Neo4j is not available, catching structural issues like unbalanced parentheses.
4. **Structured output metadata** (`query_intent`, `node_labels_used`, `relationships_used`) gives visibility into what the model generated and why.
5. **The same retry pattern** works across SQL, SPARQL, and Cypher — only the validator and schema injection change.